# IDBR Main Pipeline

End-to-end walkthrough: load/synthesize data -> preprocess -> segment -> train -> authenticate -> tamper-test -> evaluate.

Run this notebook from the `idbr_project/` root (or adjust `sys.path` below).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import torch

from src.preprocessing import load_or_synthesize, compute_normalization_stats, apply_normalization, normalize_capacity, denormalize_capacity
from src.segmentation import resample_cycles_to_windows
from src.data_loader import split_cells, filter_window_set, ContrastiveBatteryDataset, BatteryWindowDataset, contrastive_collate_fn, plain_collate_fn
from src.model import IDBRModel, BaselineModel, sigmoid_ramp
from src.losses import TotalIDBRLoss, LossWeights
from src.authentication import Enroller, FingerprintDatabase, compute_eer, compute_roc_auc, top1_retrieval_accuracy
from src.tamper import TamperEngine, generate_tampered_dataset, DEFAULT_SEVERITIES, ATTACK_TYPES
from src.train import train_idbr, build_datasets
from src.evaluate import run_full_evaluation, run_ablation_grid

## 1. Load or synthesize the NASA PCoE dataset

If `../data/raw` contains real `.mat` files they are parsed directly; otherwise a physically-motivated synthetic fallback is used so this notebook always runs end-to-end.

In [ ]:
records, used_synthetic = load_or_synthesize("../data/raw")
print(f"cells loaded: {len(records)} | used synthetic fallback: {used_synthetic}")
for cid, rec in list(records.items())[:3]:
    print(cid, "->", len(rec.cycles), "discharge cycles")

## 2. Cell-level split, resampling to T=100, and train-only normalization

In [ ]:
train_cells, val_cells, test_cells = split_cells(list(records.keys()), seed=42)
print("train cells:", train_cells)
print("val cells:  ", val_cells)
print("test cells: ", test_cells)

full_ws = resample_cycles_to_windows(records, T=100)
train_ws = filter_window_set(full_ws, train_cells)
val_ws = filter_window_set(full_ws, val_cells)
test_ws = filter_window_set(full_ws, test_cells)

stats = compute_normalization_stats(train_ws.windows, train_ws.capacities)
for ws in (train_ws, val_ws, test_ws):
    ws.windows = apply_normalization(ws.windows, stats)
    ws.capacities = normalize_capacity(ws.capacities, stats)

print("train windows:", train_ws.windows.shape, "| val:", val_ws.windows.shape, "| test:", test_ws.windows.shape)

## 3. Sanity-check the model and loss on one batch

In [ ]:
from torch.utils.data import DataLoader

num_cells = len(full_ws.label_to_int)
model = IDBRModel(num_cells=num_cells)
loss_fn = TotalIDBRLoss(temperature=0.07)

contrastive_ds = ContrastiveBatteryDataset(train_ws, seed=0)
loader = DataLoader(contrastive_ds, batch_size=16, shuffle=True, collate_fn=contrastive_collate_fn)
batch = next(iter(loader))

model.set_grl_lambda(sigmoid_ramp(0.5))
out_a = model(batch["anchor"])
out_p = model(batch["positive"])
losses = loss_fn(
    out_a["z_id"], out_p["z_id"], out_a["capacity_hat"], batch["capacity"],
    out_a["adv_logits"], batch["cell_id_int"], LossWeights(lambda1=1.0, lambda2=1.0),
)
losses

## 4. Train IDBR (short run for demonstration -- raise `epochs` for a real run)

In [ ]:
summary = train_idbr(
    raw_dir="../data/raw",
    checkpoint_dir="../checkpoints",
    results_dir="../results",
    epochs=5,
    batch_size=64,
    lambda1=1.0,
    lambda2=1.0,
    run_name="notebook_demo",
)
summary

## 5. Full evaluation suite (plots + CSV tables in `../results`)

In [ ]:
eval_summary = run_full_evaluation(
    checkpoint_path=summary["checkpoint_path"],
    raw_dir="../data/raw",
    results_dir="../results",
)
eval_summary

## 6. (Optional) lambda1 / lambda2 ablation grid (Table VI)

This retrains a small model per grid cell -- slow; skip for a quick demo.

In [ ]:
# ablation_csv = run_ablation_grid(
#     raw_dir="../data/raw", checkpoint_dir="../checkpoints", results_dir="../results",
#     lambda1_values=[0.0, 1.0], lambda2_values=[0.0, 0.5, 1.0, 2.0], epochs=10,
# )
# ablation_csv